# Pruebas de Modelo: Caso Uber Bogota

Este cuaderno compara cuatro modelos bayesianos alternativos para explicar y predecir `trips_pool`.

La idea es mantener el flujo del cuaderno original, pero separando claramente los tres enfoques recomendados para poder evaluar cual se ajusta mejor.

## Uber

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

### 1. Carga y Limpieza de Datos

Cargamos la base de Uber y construimos variables utiles para los tres modelos alternativos.

In [ ]:
df = pd.read_excel('/home/julian_jimenez/code/bayes/Taller-video-bayes/data/BaseUBER.xlsx')

df['wait_time_num'] = df['wait_time'].astype(str).str.extract(r'(\d+)').astype(float)
df['commute_num'] = df['commute'].astype(int)
df['treat_num'] = df['treat'].astype(int)

# Variables temporales derivadas de period_start
df['hour'] = df['period_start'].dt.hour + df['period_start'].dt.minute / 60
angle = 2 * np.pi * df['hour'] / 24
df['hour_sin'] = np.sin(angle)
df['hour_cos'] = np.cos(angle)

print('Primeros registros:')
display(df[['period_start', 'wait_time', 'wait_time_num', 'commute_num', 'hour', 'hour_sin', 'hour_cos', 'trips_pool']].head())

print('\nResumen estadistico de trips_pool:')
print(df['trips_pool'].describe())

### 2. Analisis Exploratorio de Datos (EDA)

Revisamos la relacion de `trips_pool` con el tiempo de espera, el indicador `commute` y la estructura horaria.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(x='wait_time_num', y='trips_pool', data=df, ax=ax[0], palette='Set2')
ax[0].set_title('Viajes segun tiempo de espera')

sns.boxplot(x='commute_num', y='trips_pool', data=df, ax=ax[1], palette='muted')
ax[1].set_title('Viajes segun commute')

sns.scatterplot(x='hour', y='trips_pool', hue='commute_num', data=df, ax=ax[2], palette='viridis')
ax[2].set_title('Viajes segun hora del dia')

plt.tight_layout()
plt.show()

### 3. Matriz de Correlacion

Incluimos las nuevas variables temporales para ver como se relacionan con `trips_pool`.

In [ ]:
numeric_cols = [
    'wait_time_num', 'commute_num', 'hour', 'hour_sin', 'hour_cos',
    'trips_pool', 'trips_express', 'rider_cancellations',
    'total_driver_payout', 'total_matches', 'total_double_matches'
]

corr = df[numeric_cols].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Matriz de correlacion de Pearson')
plt.show()

### 4. Funciones Auxiliares

Definimos una funcion para ajustar cada modelo, generar posterior predictive checks y facilitar la comparacion final.

In [ ]:
def fit_bayesian_model(data, predictors, model_name, draws=3000, tune=1000, chains=4, random_seed=42):
    with pm.Model() as model:
        beta_0 = pm.Normal('beta_0', mu=1400, sigma=500)
        sigma = pm.HalfNormal('sigma', sigma=300)

        mu = beta_0
        for predictor in predictors:
            mu = mu + pm.Normal(f'beta_{predictor}', mu=0, sigma=10) * data[predictor].values

        pm.Normal('y_obs', mu=mu, sigma=sigma, observed=data['trips_pool'].values)

        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            random_seed=random_seed,
            return_inferencedata=True,
            idata_kwargs={'log_likelihood': True}
        )

        idata.extend(pm.sample_posterior_predictive(idata, random_seed=random_seed, extend_inferencedata=False))

    return {'name': model_name, 'predictors': predictors, 'idata': idata}


def show_diagnostics(model_result):
    name = model_result['name']
    predictors = model_result['predictors']
    idata = model_result['idata']

    var_names = ['beta_0'] + [f'beta_{predictor}' for predictor in predictors] + ['sigma']

    print(f'Resumen de parametros: {name}')
    display(az.summary(idata, var_names=var_names, round_to=2))

    az.plot_trace(idata, var_names=var_names)
    plt.tight_layout()
    plt.show()

    az.plot_forest(idata, var_names=[f'beta_{predictor}' for predictor in predictors], combined=True)
    plt.axvline(0, color='red', linestyle='--')
    plt.title(f'Intervalos HDI al 95%: {name}')
    plt.show()

    az.style.use('arviz-darkgrid')
    az.plot_ppc(idata)
    plt.title(f'PPC: {name}')
    plt.show()


## Modelo Alternativo 1

Modelo explicativo base recomendado:
- `wait_time_num`
- `hour_sin`
- `hour_cos`

La idea es reemplazar `commute` por una representacion mas rica de la hora del dia.

In [ ]:
model_1 = fit_bayesian_model(
    data=df,
    predictors=['wait_time_num', 'hour_sin', 'hour_cos'],
    model_name='Modelo Alternativo 1'
)

show_diagnostics(model_1)

## Modelo Alternativo 2

Modelo explicativo con control adicional:
- `wait_time_num`
- `commute_num`
- `hour_sin`
- `hour_cos`

Aqui probamos si `commute` sigue aportando informacion una vez controlamos por la hora del dia.

In [ ]:
model_2 = fit_bayesian_model(
    data=df,
    predictors=['wait_time_num', 'commute_num', 'hour_sin', 'hour_cos'],
    model_name='Modelo Alternativo 2'
)

show_diagnostics(model_2)

## Modelo Alternativo 3

Modelo predictivo fuerte, pero menos causal:
- `wait_time_num`
- `hour_sin`
- `hour_cos`
- `total_driver_payout`
- `rider_cancellations`

Este modelo puede predecir mejor `trips_pool`, pero usa variables operativas que ocurren durante o despues del proceso.

In [ ]:
model_3 = fit_bayesian_model(
    data=df,
    predictors=['wait_time_num', 'hour_sin', 'hour_cos', 'total_driver_payout', 'rider_cancellations'],
    model_name='Modelo Alternativo 3'
)

show_diagnostics(model_3)

## Modelo Alternativo 4

Modelo reducido solicitado:
- `wait_time_num`
- `rider_cancellations`

Este modelo prueba si una especificacion mas simple, con una variable de tratamiento y una operativa, logra explicar bien `trips_pool`.

In [ ]:
model_4 = fit_bayesian_model(
    data=df,
    predictors=['wait_time_num', 'rider_cancellations'],
    model_name='Modelo Alternativo 4'
)

show_diagnostics(model_4)

## Modelo Alternativo 5

Modelo ampliado basado en el análisis de scatterplots y correlaciones con `trips_pool`:
- `wait_time_num` (r = 0.21) — variable experimental del estudio, mantiene la interpretabilidad causal.
- `commute_num` (r = 0.20) — contexto de hora pico.
- `rider_cancellations` (r = 0.37) — señal fuerte de demanda: a más demanda, más cancelaciones. Es la segunda variable con mayor correlación con `trips_pool`.
- `total_driver_payout` (r = 0.45) — variable con la correlación más alta con `trips_pool`. Refleja el volumen económico del periodo.

**Nota sobre multicolinealidad:** `rider_cancellations` y `commute_num` tienen r = 0.82 entre sí, lo que representa un riesgo de redundancia. Sin embargo, en el framework bayesiano los priors regularizadores ayudan a estabilizar los coeficientes incluso con correlación alta entre predictoras. Se incluyen ambas para evaluar si cada una aporta información independiente una vez el modelo controla por la otra.

Este es el modelo con mayor capacidad predictiva esperada de los cinco, pues combina la variable experimental con las dos variables operativas de mayor señal.

In [ ]:
model_5 = fit_bayesian_model(
    data=df,
    predictors=['wait_time_num', 'commute_num', 'rider_cancellations', 'total_driver_payout'],
    model_name='Modelo Alternativo 5'
)

show_diagnostics(model_5)

In [ ]:
comparison_dict = {
    model_1['name']: model_1['idata'],
    model_2['name']: model_2['idata'],
    model_3['name']: model_3['idata'],
    model_4['name']: model_4['idata'],
    model_5['name']: model_5['idata'],
}

compare_loo = az.compare(comparison_dict, ic='loo')
compare_waic = az.compare(comparison_dict, ic='waic')

print('Comparacion por LOO:')
display(compare_loo)

print('Comparacion por WAIC:')
display(compare_waic)

az.plot_compare(compare_loo)
plt.title('Comparacion de modelos por LOO')
plt.show()

## Conclusiones

Al finalizar la corrida del cuaderno, responde estas preguntas:

1. Cual modelo tiene mejor desempeno predictivo segun `LOO` y `WAIC`.
2. En cuales modelos los intervalos HDI de las betas no cruzan cero.
3. Si el mejor modelo tambien es el mas interpretable para el objetivo del taller.
4. Si conviene priorizar un modelo explicativo o uno predictivo.
5. **Modelo 5:** ¿Agregar `total_driver_payout` y `rider_cancellations` junto a `wait_time_num` y `commute_num` mejoró significativamente el LOO respecto a los modelos anteriores? ¿Alguna de las cuatro betas tiene un HDI que no cruce cero, indicando un efecto claro? ¿La alta correlación entre `rider_cancellations` y `commute_num` (r = 0.82) se refleja en inestabilidad de las cadenas MCMC (traceplot ruidoso) o los priors regularizaron bien?